imports

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '../..')))

from ClassificationModel.src.models.classification_model import CancerClassificationModel
from ClassificationModel.src.utils.dataset_utils import load_dataset
from constants.classification.datasets_constants import DatasetConstants
from constants.classification.model_constants import ModelConstants
from utils.callbacks.notification_callback import NotificationCallback
from tensorflow import keras
from ClassificationModel.src.data_processing.image_augmentation import ImageAugmentationPipeline

/Users/ilayn/Ort/FinalsProject/MachineLearning/.venv/lib/python3.13/site-packages/keras/src/export/tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):
/Users/ilayn/Ort/FinalsProject/MachineLearning/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading the dataset and creating the model

In [2]:
dataset = load_dataset()

model = CancerClassificationModel(dataset, (*DatasetConstants.IMAGE_SIZE, DatasetConstants.CHANNELS))

augmenter = ImageAugmentationPipeline()

EPOCHS=50

Found 1013 files belonging to 4 classes.
Found 144 files belonging to 4 classes.
Found 610 files belonging to 4 classes.


defining callbacks

In [ ]:
callbacks = [
    NotificationCallback(
        notifier=model.notifier,
        metrics_to_track=[
            ModelConstants.TRACKED_ACCURACY_METRIC,
            ModelConstants.TRACKED_LOSS_METRIC,
            ModelConstants.TRACKED_VAL_ACCURACY_METRIC,
            ModelConstants.TRACKED_VAL_LOSS_METRIC
        ]),
    keras.callbacks.ModelCheckpoint(
        filepath=f'{ModelConstants.CHECKPOINT_DIR_PATH}/best_model.keras',
        save_best_only=True,
        monitor='val_loss'
    ),
    keras.callbacks.EarlyStopping(
        monitor=ModelConstants.TRACKED_VAL_LOSS_METRIC,
        patience=15, 
        restore_best_weights=True
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor=ModelConstants.TRACKED_VAL_LOSS_METRIC,
        factor=0.5,
        patience=5,  
        min_lr=1e-7
    )
]

training the model

In [4]:
history = model.train_model(epochs=EPOCHS, callbacks=callbacks,augment_train=True,augmenter=augmenter)

Epoch 1/50
32/32 ━━━━━━━━━━━━━━━━━━━━ 52s 2s/step - accuracy: 0.7957 - auc: 0.9485 - loss: 0.5526 - precision: 0.8509 - recall: 0.7660 - val_accuracy: 0.1806 - val_auc: 0.5122 - val_loss: 5.0233 - val_precision: 0.1831 - val_recall: 0.1806 - learning_rate: 0.0010
Epoch 2/50
32/32 ━━━━━━━━━━━━━━━━━━━━ 50s 2s/step - accuracy: 0.9348 - auc: 0.9905 - loss: 0.2309 - precision: 0.9415 - recall: 0.9210 - val_accuracy: 0.5139 - val_auc: 0.7725 - val_loss: 1.1591 - val_precision: 0.6538 - val_recall: 0.4722 - learning_rate: 0.0010
Epoch 3/50
32/32 ━━━━━━━━━━━━━━━━━━━━ 52s 2s/step - accuracy: 0.9358 - auc: 0.9935 - loss: 0.1915 - precision: 0.9456 - recall: 0.9260 - val_accuracy: 0.6667 - val_auc: 0.8572 - val_loss: 0.9878 - val_precision: 0.7143 - val_recall: 0.6250 - learning_rate: 0.0010
Epoch 4/50
32/32 ━━━━━━━━━━━━━━━━━━━━ 52s 2s/step - accuracy: 0.9743 - auc: 0.9981 - loss: 0.0835 - precision: 0.9772 - recall: 0.9743 - val_accuracy: 0.8472 - val_auc: 0.9513 - val_loss: 0.5231 - val_precisi

evaluating the model

In [5]:

results = model.evaluate_model(present_metrics=True,send_message=True)

20/20 ━━━━━━━━━━━━━━━━━━━━ 9s 446ms/step - accuracy: 0.6049 - auc: 0.7974 - loss: 1.6172 - precision: 0.6298 - recall: 0.5885
accuracy: 0.605
auc: 0.797
loss: 1.617
precision: 0.630
recall: 0.589
